In [1]:
# Import libraris
from collections import Counter # type: ignore
import json # type: ignore
import torch # type: ignore
import torch.nn as nn # type: ignore
from torch.utils.data import Dataset # type: ignore
import torch.utils.data # type: ignore
import math # type: ignore
import torch.nn.functional as f # type: ignore
import torch_directml # type: ignore

In [2]:
# Define device
device = torch_directml.device(0)
print(torch_directml.device_name(0))

AMD Radeon RX 6800S 


In [3]:
# Necessary constants
corpus_movie_conv = './source/cornell_movie_dialogs_corpus/cornell_movie_dialogs_corpus/movie_conversations.txt'
corpus_movie_lines = './source/cornell_movie_dialogs_corpus/cornell_movie_dialogs_corpus/movie_lines.txt'
MAX_LENGTH = 25

In [4]:
# Open files and save data
with open(corpus_movie_conv, 'r', encoding='ISO-8859-1') as c:
    conversations = c.readlines()
with open(corpus_movie_lines, 'r', encoding='ISO-8859-1') as l:
    lines = l.readlines()

In [5]:
conversations[:4]

["u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L194', 'L195', 'L196', 'L197']\n",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L198', 'L199']\n",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L200', 'L201', 'L202', 'L203']\n",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L204', 'L205', 'L206']\n"]

In [6]:
lines[:4]

['L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!\n',
 'L1044 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ They do to!\n',
 'L985 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ I hope so.\n',
 'L984 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ She okay?\n']

In [7]:
# Create a dictionary of lines
lines_dictionary = {}
for line in lines:
    objects = line.split(' +++$+++ ')
    lines_dictionary[objects[0]] = objects[-1]

In [8]:
list(lines_dictionary.items())[:4]

[('L1045', 'They do not!\n'),
 ('L1044', 'They do to!\n'),
 ('L985', 'I hope so.\n'),
 ('L984', 'She okay?\n')]

In [9]:
# Function that removes punctuatoin
punctuations = {'!', '(', ')', '-', '[', ']', '{', '}', ';', ':', '\'', '"', '\\', ',',
                    '<', '>', '.', '/', '?', '@', '#', '$', '%', '^', '&', '*', '_', '~'}
def remove_punctuation(string):
    new_string = [char for char in string if char not in punctuations]
    return ''.join(new_string).lower()

In [10]:
remove_punctuation("Hello, Nigga!")

'hello nigga'

In [11]:
# Create pairs of dialogs
pairs = []
for conversation in conversations:
    indexes = eval(conversation.split(' +++$+++ ')[-1])
    for i in range(len(indexes) - 1):
        qa_pairs = []
        first = remove_punctuation(lines_dictionary[indexes[i]]).strip()
        second = remove_punctuation(lines_dictionary[indexes[i + 1]]).strip()
        qa_pairs.append(first.split()[:MAX_LENGTH])
        qa_pairs.append(second.split()[:MAX_LENGTH])
        pairs.append(qa_pairs)

In [12]:
pairs[0]

[['can',
  'we',
  'make',
  'this',
  'quick',
  'roxanne',
  'korrine',
  'and',
  'andrew',
  'barrett',
  'are',
  'having',
  'an',
  'incredibly',
  'horrendous',
  'public',
  'break',
  'up',
  'on',
  'the',
  'quad',
  'again'],
 ['well',
  'i',
  'thought',
  'wed',
  'start',
  'with',
  'pronunciation',
  'if',
  'thats',
  'okay',
  'with',
  'you']]

In [13]:
len(pairs)

221616

In [14]:
# Calculate word frequencies
word_frequency = Counter()
for pair in pairs:
    word_frequency.update(pair[0])
    word_frequency.update(pair[1])

In [15]:
word_frequency.most_common(4)

[('you', 169695), ('i', 137639), ('the', 120903), ('to', 100667)]

In [16]:
# Create a list of all words and vocabulary
min_frequency = 5
words = [w for w, c in word_frequency.items() if c > min_frequency]
word2index = {k : v + 1 for v, k in enumerate(words)}
word2index['<unk>'] = len(word2index) + 1
word2index['<start>'] = len(word2index) + 1
word2index['<end>'] = len(word2index) + 1
word2index['<pad>'] = 0

In [17]:
print(len(word2index))

18243


In [18]:
# Saving vocabulary
with open('./source/cornell_movie_dialogs_corpus/WORD2INDEX.json', 'w') as j:
    json.dump(word2index, j)

In [19]:
# Encode question function
def encode_question(words, word2index):
    encoded = ([word2index.get(word, word2index['<unk>']) for word in words] +
               [word2index['<pad>']] * (MAX_LENGTH - len(words)))
    return encoded

In [20]:
# Encode reply function
def encode_reply(words, word2index):
    encoded = ([word2index['<start>']] +
               [word2index.get(word, word2index['<unk>']) for word in words] +
               [word2index['<end>']] +
               [word2index['<pad>']] * (MAX_LENGTH - len(words)))
    return encoded

In [21]:
encode_question(pairs[0][0], word2index)

[1,
 2,
 3,
 4,
 5,
 18240,
 18240,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 18240,
 13,
 14,
 15,
 16,
 17,
 18240,
 18,
 0,
 0,
 0]

In [22]:
encode_reply(pairs[0][1], word2index)

[18241,
 19,
 20,
 21,
 22,
 23,
 24,
 18240,
 25,
 26,
 27,
 24,
 28,
 18242,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [23]:
# Encode pairs
pairs_encoded = []
for pair in pairs:
    question = encode_question(pair[0], word2index)
    reply = encode_reply(pair[1], word2index)
    pairs_encoded.append([question, reply])

In [24]:
pairs_encoded[0]

[[1,
  2,
  3,
  4,
  5,
  18240,
  18240,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  18240,
  13,
  14,
  15,
  16,
  17,
  18240,
  18,
  0,
  0,
  0],
 [18241,
  19,
  20,
  21,
  22,
  23,
  24,
  18240,
  25,
  26,
  27,
  24,
  28,
  18242,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]]

In [25]:
# Save encoded pairs
with open('./source/cornell_movie_dialogs_corpus/pairs_encoded.json', 'w') as j:
    json.dump(pairs_encoded, j)

In [26]:
# Create custom dataset
class Dataset(Dataset):
    def __init__(self):
        self.pairs = json.load(open('./source/cornell_movie_dialogs_corpus/pairs_encoded.json'))
        self.dataset_size = len(self.pairs)
    def __getitem__(self, index):
        question = torch.LongTensor(self.pairs[index][0])
        reply = torch.LongTensor(self.pairs[index][1])
        return question, reply
    def __len__(self):
        return self.dataset_size

In [27]:
# Create data loader
train_loader = torch.utils.data.DataLoader(Dataset(),
                                           batch_size=100,
                                           shuffle=True,
                                           pin_memory=True)

In [28]:
# Create masks function
def create_mask(question, reply_input, reply_target):
    def subsequent_mask(size):
        return torch.tril(torch.ones(size, size)).type(dtype=torch.bool).unsqueeze(0)
    question_mask = (question != 0).to(device).unsqueeze(1).unsqueeze(1)
    reply_input_mask = (reply_input != 0).unsqueeze(1)
    reply_input_mask = (reply_input_mask & subsequent_mask(reply_input.size(-1)).type_as(reply_input_mask.data))
    reply_input_mask = reply_input_mask.unsqueeze(1)
    reply_target_mask = (reply_target != 0)
    return question_mask, reply_input_mask, reply_target_mask

In [29]:
# Embedding class with positional encoding
class Embedding(nn.Module):
    def __init__(self, vocabulary_size, d_model, max_len=50, n_layers=6):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(p=0.1)
        self.embed = nn.Embedding(vocabulary_size, d_model)
        self.pe = self.create_positional_encoding(max_len, d_model)
        self.te = self.create_positional_encoding(n_layers, d_model)
    def create_positional_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model).to(device)
        for pos in range(max_len):
            for i in range(0, d_model - 1, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (i / d_model)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** (i / d_model)))
        pe = pe.unsqueeze(0)
        return pe
    def forward(self, embeddings, layer_index):
        if layer_index == 0:
            embeddings = self.embed(embeddings) * math.sqrt(self.d_model)
        embeddings += self.pe[:, :embeddings.size(1), :]
        embeddings += self.te[:, layer_index, :].unsqueeze(1)
        embeddings = self.dropout(embeddings)
        return embeddings

In [30]:
# Multi Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, heads, d_model):
        super().__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.heads = heads
        self.dropout = nn.Dropout(p=0.1)
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.concat = nn.Linear(d_model, d_model)
    def forward(self, query, key, value, mask):
        query = self.query(query)
        key = self.key(key)
        value = self.value(value)
        query = query.view(query.size(0), -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        key = key.view(key.size(0), -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        value = value.view(value.size(0), -1, self.heads, self.d_k).permute(0, 2, 1, 3)
        scores = torch.matmul(query, key.permute(0, 1, 3, 2)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(mask == 0, -1e9)
        weights = f.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        context = torch.matmul(weights, value)
        context = context.permute(0, 2, 1, 3).reshape(context.size(0), -1, self.heads * self.d_k)
        context = self.concat(context)
        return context

In [31]:
# Feed Forward
class FeedForward(nn.Module):
    def __init__(self, d_model, middle_dim = 2048):
        super().__init__()
        self.fc1 = nn.Linear(d_model, middle_dim)
        self.fc2 = nn.Linear(middle_dim, d_model)
        self.dropout = nn.Dropout(p=0.1)
    def forward(self, x):
        x = self.fc1(x)
        x = f.relu(x)
        x = self.dropout(x)
        out = self.fc2(x)
        return out

In [32]:
# Encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.self_multihead = MultiHeadAttention(heads, d_model)
        self.feed_forward = FeedForward(d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=0.1)
    def forward(self, embeddings, mask):
        interacted = self.self_multihead(embeddings, embeddings, embeddings, mask)
        interacted = self.dropout(interacted)
        interacted += embeddings
        interacted = self.layer_norm(interacted)
        feed_forward_out = self.feed_forward(interacted)
        feed_forward_out = self.dropout(feed_forward_out)
        feed_forward_out += interacted
        encoded = self.layer_norm(feed_forward_out)
        return encoded

In [33]:
# Decoder layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.self_multihead = MultiHeadAttention(heads, d_model)
        self.source_multihead = MultiHeadAttention(heads, d_model)
        self.feed_forward = FeedForward(d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=0.1)
    def forward(self, embeddings, encoded, source_mask, target_mask):
        query = self.self_multihead(embeddings, embeddings, embeddings, target_mask)
        query = self.dropout(query)
        query += embeddings
        query = self.layer_norm(query)
        interacted = self.source_multihead(query, encoded, encoded, source_mask)
        interacted = self.dropout(interacted)
        interacted += query
        interacted = self.layer_norm(interacted)
        feed_forward_out = self.feed_forward(interacted)
        feed_forward_out = self.dropout(feed_forward_out)
        feed_forward_out += interacted
        decoded = self.layer_norm(feed_forward_out)
        return decoded

In [34]:
# Class for transformer
class Transformer(nn.Module):
    def __init__(self, d_module, heads, num_layers, word_map):
        super().__init__()
        self.d_model = d_module
        self.heads = heads
        self.num_layers = num_layers
        self.vocabulary_size = len(word_map)
        self.embed = Embedding(self.vocabulary_size, self.d_model, n_layers=num_layers)
        self.encoder = EncoderLayer(self.d_model, self.heads)
        self.decoder = DecoderLayer(self.d_model, self.heads)
        self.logit = nn.Linear(self.d_model, self.vocabulary_size)
    def encode(self, source_embeddings, source_mask):
        for i in range(self.num_layers):
            source_embeddings = self.embed(source_embeddings, i)
            source_embeddings = self.encoder(source_embeddings, source_mask)
        return source_embeddings
    def decode(self, target_embeddings, target_mask, source_embeddings, source_mask):
        for i in range(self.num_layers):
            target_embeddings = self.embed(target_embeddings, i)
            target_embeddings = self.decoder(target_embeddings, source_embeddings, source_mask, target_mask)
        return target_embeddings
    def forward(self, source_words, source_mask, target_words, target_mask):
        encoded = self.encode(source_words, source_mask)
        decoded = self.decode(target_words, target_mask, encoded, source_mask)
        out = self.logit(decoded)
        out = f.log_softmax(out, dim=-1)
        return out

In [35]:
# Optimizer with warmup
class AdamWarmup:
    def __init__(self, d_model, warmup_steps, optimzer):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.optimizer = optimzer
        self.current_step = 0
        self.learning_rate = 0
    def get_learning_rate(self):
        return self.d_model  ** (-0.5) * min(self.current_step ** (-0.5),
                                             self.current_step * self.warmup_steps ** (-1.5))
    def step(self):
        self.current_step += 1
        learning_rate = self.get_learning_rate()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = learning_rate
        self.learning_rate = learning_rate
        self.optimizer.step()

In [36]:
# Loss
class Loss(nn.Module):
    def __init__(self, vocabulary_size, smooth):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction='none')
        self.confidence = 1.0 - smooth
        self.smooth = smooth
        self.vocabulary_size = vocabulary_size
    def forward(self, prediction, target, mask):
        prediction = prediction.view(-1, self.vocabulary_size)
        target = target.reshape(-1)
        mask = mask.float()
        mask = mask.reshape(-1)
        labels = prediction.data.clone()
        labels.fill_(self.smooth / (self.vocabulary_size - 1))
        labels.scatter_(1, target.data.unsqueeze(1), self.confidence)
        loss = self.criterion(prediction, labels)
        loss = (loss.sum(1) * mask).sum() / mask.sum()
        return loss

In [37]:
# Define hyperparameters
d_model = 512
heads = 8
num_layers = 6
num_epochs = 25
with open('./source/cornell_movie_dialogs_corpus/WORD2INDEX.json', 'r') as j:
    word_wap = json.load(j)
transformer = Transformer(d_model, heads, num_layers, word_wap)
transformer = transformer.to(device)
adam_optimizer = torch.optim.Adam(transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
transformer_optimizer = AdamWarmup(d_model, 4000, adam_optimizer)
criterion = Loss(len(word_wap), smooth=0.3)

In [38]:
# Training function
def train(train_loader, model, criterion, epoch):
    model.train()
    sum_loss = 0
    count = 0
    for i, (question, reply) in enumerate(train_loader):
        batch_size = question.size(0)
        question = question.to(device)
        reply = reply.to(device)
        reply_input = reply[:, : -1]
        reply_target = reply[:, 1 :]
        question_mask, reply_input_mask, reply_target_mask = create_mask(question,
                                                                         reply_input,
                                                                         reply_target)
        out = model(question, question_mask, reply_input, reply_input_mask)
        loss = criterion(out, reply_target, reply_target_mask)
        transformer_optimizer.optimizer.zero_grad()
        loss.backward()
        transformer_optimizer.step()

        sum_loss += loss.item() * batch_size
        count += batch_size
        if i % 100 == 0:
            print(f"Epoch: {epoch}/{num_epochs}")
            print(f"Iteration: {i}/{len(train_loader)}")
            print(f"Loss: {sum_loss * 1.0 / count:.3f}")
            print()

In [39]:
# Evaluation function
def evaluate(model, question, question_mask, max_length, word_map):
    model.eval()
    reversed_word_map = {value : key for key, value in word_map.items()}
    token = word_map['<start>']
    encoded = model.encode(question, question_mask)
    words = torch.LongTensor([[token]]).to(device)
    for step in range(max_length - 1):
        size = words.size(1)
        target_mask = torch.tril(torch.ones(size, size)).type(dtype=torch.uint8)
        target_mask = target_mask.to(device).unsqueeze(0).unsqueeze(0)
        decoded = model.decode(words, target_mask, encoded, question_mask)
        predictions = model.logit(decoded[:, -1])
        probs = torch.softmax(predictions / 0.7, dim=-1)
        next_word = torch.multinomial(probs, num_samples=1)
        new_word = next_word.item()
        if new_word == word_map['<end>']:
            break
        words = torch.cat([words, torch.LongTensor([[next_word]]).to(device)], dim=1)
    words = words.squeeze(0)
    words = words.tolist()
    sentence_indexes = [w for w in words if w not in {word_map['<start>'],
                                                      word_map['<end>'],
                                                      word_map['<pad>']}]
    sentence = [reversed_word_map[w] for w in sentence_indexes]
    return ' '.join(sentence)

In [40]:
# Train
for epoch in range(num_epochs):
    train(train_loader, transformer, criterion, epoch)
    state = {
        'epoch': epoch,
        'transformer': transformer,
        'optimizer': transformer_optimizer
    }
    torch.save(state, f"./source/checkpoint_{epoch}.pth.tar")

C:\Alexey\Projects\Udemy\NN_learning\.venv\Lib\site-packages\torch\nn\functional.py:3006: UserWarning: The operator 'aten::xlogy.OutTensor' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  reduced = torch.kl_div(input, target, reduction_enum, log_target=log_target)


Epoch: 0/25
Iteration: 0/2217
Loss: 6.534



KeyboardInterrupt: 

In [69]:
# Load model
checkpoint = torch.load(f"./source/checkpoint_{9}.pth.tar")
transformer = checkpoint['transformer']

C:\Users\korol\AppData\Local\Temp\ipykernel_7792\3881797594.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f"./source/checkpoint_{9}.pth.tar")


In [70]:
while (1):
    question = input('Question: ')
    if question == 'exit':
        break
    else:
        print('Question: ', question)
        max_len = 20
        question = [word2index.get(word, word2index['<unk>'])
                            for word in remove_punctuation(question).split()]
        question = torch.LongTensor(question).to(device).unsqueeze(0)
        question_mask = (question != 0).to(device).unsqueeze(1).unsqueeze(1)
        reply = evaluate(transformer, question, question_mask, int(max_len), word2index)
        print('Reply: ', reply)

Question:  Hello
Reply:  i gave you a break of heart
Question:  how are you
Reply:  why tennis city
Question:  can you say somthing
Reply:  <unk>
Question:  why are you talking so strange
Reply:  you know we just cant afford a little bit of a thing
Question:  your phrasis are too disconected
Reply:  i dont know
Question:  are you able to kill a person?
Reply:  for the rest of the world and the captain <unk> soft with the job
Question:  can you talk normally?
Reply:  i want to ask you a question
Question:  what question?
Reply:  why are you so upset
Question:  i am disappointed because of your bad answers
Reply:  yes i do
Question:  i am as well
Reply:  yes
Question:  this is not funny. i spend so much time working on you
Reply:  just think its true
Question:  but your responses are  so bad
Reply:  so paperwork and ready for this
Question:  i did everything i could
Reply:  no
Question:  yes i did
Reply:  i dont know
Question:  i know i did
Reply:  yeah im just kidding
Question:  this is